<a href="https://colab.research.google.com/github/Musharrafmrm/hybrid-ecommerce-issue-detection/blob/main/notebooks/09_Hybrid_Model_Training_Final_Part_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# 09_Hybrid_Model_Training_Final.ipynb
# COMPLETE HYBRID MODEL (Self-Contained - No previous files needed)
# =============================================================================

import pandas as pd
import re
import os
import zipfile
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import joblib

nltk.download('stopwords', quiet=True)

print("🚀 Starting Full Hybrid Model Pipeline...\n")

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ====================== 1. Download + Clean Data ======================
if not os.path.exists('data/Reviews.csv'):
    print("Downloading Amazon dataset...")
    !kaggle datasets download -d snap/amazon-fine-food-reviews -f Reviews.csv --path data/ --unzip
    if os.path.exists('data/Reviews.csv.zip'):
        with zipfile.ZipFile('data/Reviews.csv.zip', 'r') as z:
            z.extractall('data/')
    print("✅ Dataset downloaded and unzipped")

df = pd.read_csv('data/Reviews.csv')
print(f"Original rows: {len(df):,}")

# Keep useful columns and clean
df = df[['Score', 'Summary', 'Text']].copy()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['Text'].apply(clean_text)
df = df[df['cleaned_text'].str.len() > 15]

df.to_csv('data/cleaned_dataset.csv', index=False)
print(f"After cleaning: {len(df):,} reviews")

# ====================== 2. LDA (Aspect Discovery) ======================
print("\nTraining LDA...")
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)
X_text = vectorizer.fit_transform(df['cleaned_text'])

lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X_text)

joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("✅ LDA completed")

# Add LDA features
topic_dist = lda.transform(X_text)
for i in range(5):
    df[f'topic_{i+1}'] = topic_dist[:, i]

# ====================== 3. Hybrid Model Training ======================
print("\nPreparing features for hybrid model...")
df['label'] = df['Score'].apply(lambda x: 0 if x <= 3 else 1)   # 0 = Issue, 1 = No Issue

lda_cols = [col for col in df.columns if col.startswith('topic_')]
X = df[lda_cols].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

print("\nTraining SVM...")
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# Save models
joblib.dump(rf, 'models/random_forest_hybrid.pkl')
joblib.dump(svm, 'models/svm_hybrid.pkl')

print("\n✅ Hybrid models saved successfully!")
print("\n🎉 RESEARCH MODEL TRAINING COMPLETED!")

🚀 Starting Full Hybrid Model Pipeline...

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 115M/115M [00:00<00:00, 176MB/s] 

✅ Dataset downloaded and unzipped
Original rows: 568,454
After cleaning: 568,452 reviews

Training LDA...
✅ LDA completed

Preparing features for hybrid model...

Training Random Forest...
Random Forest Accuracy: 0.8551688348242165
              precision    recall  f1-score   support

           0       0.78      0.47      0.58     24787
           1       0.87      0.96      0.91     88904

    accuracy                           0.86    113691
   macro avg       0.82      0.72      0.75    113691
weighted avg       0.85      0.86      0.84    113691


Training SVM...
SVM Accuracy: 0.7819792243889138
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     24787
           1       0.78      1.00      0.88     88904

    accuracy                           0.78    113

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



✅ Hybrid models saved successfully!

🎉 RESEARCH MODEL TRAINING COMPLETED!
